# SDTM Test Codes — Diff vs pre-2026-03 baseline

Compares the freshly regenerated `SDTM_Test_Identity.xlsx` and `SDTM_Instrument_Test_Identity.xlsx`
against the snapshots saved under `machine_actionable/pre-2026-03/` before bumping to the
SDTM CT 2026-03 package.

Key: `NCIt_Code` (unique per row in both artifacts).
Sheet: `Test Codes` with header on row 2 (skiprows=1).

Outputs three classifications per artifact: **added**, **removed**, **changed** rows.


In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..') / 'machine_actionable'
BASELINE = ROOT / 'pre-2026-03'
KEY = 'NCIt_Code'
SHEET = 'Test Codes'

# (current_filename, baseline_filename) — the instrument test identity file
# was renamed in April 2026; the pre-2026-03 baseline keeps its historical name.
ARTIFACTS = [
    ('SDTM_Test_Identity.xlsx',            'SDTM_Test_Identity.xlsx'),
    ('SDTM_Instrument_Test_Identity.xlsx', 'SDTM_Instrument_Identity.xlsx'),
]


In [ ]:
def load(path):
    df = pd.read_excel(path, sheet_name=SHEET, header=0)
    df = df.dropna(subset=[KEY]).reset_index(drop=True)
    return df

def diff(old, new, key=KEY):
    old_keys = set(old[key])
    new_keys = set(new[key])
    added = new[new[key].isin(new_keys - old_keys)].copy()
    removed = old[old[key].isin(old_keys - new_keys)].copy()
    common = sorted(old_keys & new_keys)
    o = old[old[key].isin(common)].set_index(key).sort_index()
    n = new[new[key].isin(common)].set_index(key).sort_index()
    cols = [c for c in o.columns if c in n.columns]
    o = o[cols]; n = n[cols]
    # normalize NaN-vs-NaN to equal
    diff_mask = (o.fillna('').astype(str) != n.fillna('').astype(str))
    changed_keys = diff_mask.any(axis=1)
    changed_rows = []
    for k in o.index[changed_keys]:
        for c in cols:
            ov, nv = o.at[k, c], n.at[k, c]
            if str(ov) if pd.notna(ov) else '' != (str(nv) if pd.notna(nv) else ''):
                if (pd.isna(ov) and pd.isna(nv)):
                    continue
                if ov != nv:
                    changed_rows.append({KEY: k, 'column': c, 'old': ov, 'new': nv})
    changed = pd.DataFrame(changed_rows)
    return added, removed, changed


In [ ]:
results = {}
for current_name, baseline_name in ARTIFACTS:
    old = load(BASELINE / baseline_name)
    new = load(ROOT / current_name)
    added, removed, changed = diff(old, new)
    results[current_name] = (old, new, added, removed, changed)
    print(f'=== {current_name} ===')
    print(f'  baseline rows: {len(old)}')
    print(f'  new rows:      {len(new)}')
    print(f'  added:         {len(added)}')
    print(f'  removed:       {len(removed)}')
    print(f'  changed cells: {len(changed)} (across {changed[KEY].nunique() if len(changed) else 0} rows)')


## Inspect: SDTM_Test_Identity.xlsx


In [ ]:
art = 'SDTM_Test_Identity.xlsx'
_, _, added, removed, changed = results[art]
added.head(20)


In [ ]:
removed.head(20)


In [ ]:
changed.head(40)


## Inspect: SDTM_Instrument_Test_Identity.xlsx


In [ ]:
art = 'SDTM_Instrument_Test_Identity.xlsx'
_, _, added, removed, changed = results[art]
added.head(20)


In [ ]:
removed.head(20)


In [ ]:
changed.head(40)


## Optional: write diff CSVs

Uncomment to persist the diff outputs alongside the notebook for review/commit.


In [ ]:
OUT = Path('..') / 'machine_actionable' / 'diffs' / 'pre-2026-03__current'
OUT.mkdir(parents=True, exist_ok=True)
for art, (_, _, added, removed, changed) in results.items():
    stem = Path(art).stem
    added.to_csv(OUT / f'{stem}__added.csv', index=False)
    removed.to_csv(OUT / f'{stem}__removed.csv', index=False)
    changed.to_csv(OUT / f'{stem}__changed.csv', index=False)
print('wrote', OUT)
